# Eval 120 Stage2 Sequential Inference (Disk-safe)

This notebook runs **one model at a time** for `eval_120.csv` and deletes cache after each model.

- Prompt format: `"<EN> {source_text} <KO>"`
- Generation: `max_new_tokens=1024`, `do_sample=False`
- Input truncation: **disabled**
- Output schema: `item_id,source_text,hypothesis`


In [ ]:
# Runtime bootstrap: install packages into the active notebook kernel env
# (vast.ai / Lightning AI / Colab compatible)
import subprocess
import sys


def pip_install(args, optional=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(args)
    print('running:', ' '.join(cmd))
    try:
        subprocess.check_call(cmd)
        return True
    except Exception as e:
        if optional:
            print(f'[WARN] optional install failed: {args} -> {type(e).__name__}: {e}')
            return False
        raise


# Core stack (as requested)
pip_install([
    'unsloth',
    'unsloth-zoo',
    'datasets',
    'trl',
    'hydra-core',
    'omegaconf',
    'sentencepiece',
    'numpy',
])

# Eval / utility stack (as requested)
pip_install([
    'lm-eval',
    'mergekit',
    'pandas',
    'tabulate',
    'matplotlib',
])

# Optional speed stack: xformers wheel may be unavailable on some CUDA/Torch combos
XFORMERS_INSTALL_OK = pip_install(['xformers'], optional=True)
print(f'xformers_install_ok={XFORMERS_INSTALL_OK}')

print('bootstrap install done')

# (Optional) HF login for private repos
# from huggingface_hub import notebook_login; notebook_login()



In [ ]:
%pip install -U "transformers==5.5.4" "trl>=0.15.0" --no-deps
import transformers
print(transformers.__version__)


In [ ]:
import gc
import os
import re
import shutil
from pathlib import Path
import importlib.util

import pandas as pd
import torch

# Check install only; avoid early import so Gemma can run without Unsloth global patching.
HAS_UNSLOTH = importlib.util.find_spec('unsloth') is not None
if not HAS_UNSLOTH:
    print('[INFO] unsloth not installed; qwen will use transformers fallback.')

try:
    import xformers  # noqa: F401
    HAS_XFORMERS = True
except Exception as e:
    HAS_XFORMERS = False
    print(f'[INFO] xformers import failed or unavailable: {type(e).__name__}: {e}')

from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

try:
    from transformers import AutoModelForImageTextToText
    HAS_AUTO_MODEL_FOR_IMAGE_TEXT_TO_TEXT = True
except Exception:
    AutoModelForImageTextToText = None
    HAS_AUTO_MODEL_FOR_IMAGE_TEXT_TO_TEXT = False

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
print(f"has_unsloth={HAS_UNSLOTH} (lazy import)")
print(f"has_xformers={HAS_XFORMERS}")
print(f"has_auto_model_for_image_text_to_text={HAS_AUTO_MODEL_FOR_IMAGE_TEXT_TO_TEXT}")
if torch.cuda.is_available():
    print(f"cuda_device={torch.cuda.get_device_name(0)}")



In [ ]:
# Paths & run config
INPUT_CSV = Path('/workspace/eval_120.csv')
OUTPUT_DIR = Path('/workspace/outputs/predictions')
HF_CACHE_ROOT = Path('/workspace/hf_tmp')

STAGE2_MODELS = [
    'alwaysgood/gemma-4-E2B-stage2',
    'alwaysgood/gemma4-CPT-stage2',
    'alwaysgood/QWEN3-4B-Base-stage2',
    'alwaysgood/QWEN3-4B-CPT-stage2',
    'alwaysgood/QWEN3.5-4B-Base-stage2',
    'alwaysgood/QWEN3.5-4B-CPT-half-lr-stage2',
]

BASELINE_MODELS = [
    'unsloth/Qwen3-4B-Base',
    'unsloth/Qwen3.5-4B-Base',
    'unsloth/gemma-4-E2B',
]

RUN_STAGE2_MODELS = True
RUN_BASELINE_MODELS = True
MODELS = []
if RUN_STAGE2_MODELS:
    MODELS.extend(STAGE2_MODELS)
if RUN_BASELINE_MODELS:
    MODELS.extend(BASELINE_MODELS)

PROMPT_TEMPLATE = '<EN> {source_text} <KO>'
MAX_NEW_TOKENS = 1024
DO_SAMPLE = False
TEMPERATURE = 0.0

# Speed config
BATCH_SIZE_QWEN = 8
BATCH_SIZE_GEMMA = 8
SORT_BY_INPUT_LENGTH = True
ENABLE_TF32 = True
FORCE_BF16 = True
ENABLE_FLASH_ATTN_FOR_TRANSFORMERS = True
ENABLE_XFORMERS_FOR_TRANSFORMERS = True
USE_UNSLOTH_FAST_INFERENCE = True

# Unsloth loading config
USE_UNSLOTH = bool(HAS_UNSLOTH)
UNSLOTH_MAX_SEQ_LENGTH = 4096
FORCE_TRANSFORMERS_FOR_GEMMA = True

# Optional smoke-test settings
RUN_SMOKE_TEST = False
SMOKE_LIMIT = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print(f'INPUT_CSV={INPUT_CSV}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')
print(f'HF_CACHE_ROOT={HF_CACHE_ROOT}')
print(f'USE_UNSLOTH={USE_UNSLOTH}')
print(f'FORCE_TRANSFORMERS_FOR_GEMMA={FORCE_TRANSFORMERS_FOR_GEMMA}')
print(f'RUN_STAGE2_MODELS={RUN_STAGE2_MODELS}, count={len(STAGE2_MODELS)}')
print(f'RUN_BASELINE_MODELS={RUN_BASELINE_MODELS}, count={len(BASELINE_MODELS)}')
print(f'TOTAL_MODELS={len(MODELS)}')
print(f'BATCH_SIZE_QWEN={BATCH_SIZE_QWEN}, BATCH_SIZE_GEMMA={BATCH_SIZE_GEMMA}')
print(f'SORT_BY_INPUT_LENGTH={SORT_BY_INPUT_LENGTH}')
print(f'ENABLE_FLASH_ATTN_FOR_TRANSFORMERS={ENABLE_FLASH_ATTN_FOR_TRANSFORMERS}')
print(f'ENABLE_XFORMERS_FOR_TRANSFORMERS={ENABLE_XFORMERS_FOR_TRANSFORMERS}')
print(f'FORCE_BF16={FORCE_BF16}')

if ENABLE_TF32 and torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.allow_tf32 = True
    print('[speed] TF32 enabled')

if not USE_UNSLOTH:
    print('[WARN] unsloth unavailable; qwen will use transformers fallback.')


In [ ]:
# Load input CSV and validate columns
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')

raw_df = pd.read_csv(INPUT_CSV)
cols = set(raw_df.columns)
if 'source_text' not in cols:
    raise ValueError(f"Missing required column 'source_text'. available={list(raw_df.columns)}")

if 'item_id' in cols:
    item_col = 'item_id'
elif 'item_number' in cols:
    item_col = 'item_number'
else:
    raise ValueError("Missing item id column. Need one of ['item_id', 'item_number']")

eval_df = raw_df[[item_col, 'source_text']].copy()
eval_df = eval_df.rename(columns={item_col: 'item_id'})
eval_df['item_id'] = eval_df['item_id'].astype(str).str.strip()
eval_df['source_text'] = eval_df['source_text'].fillna('').astype(str)
eval_df = eval_df[eval_df['item_id'].str.len() > 0].copy()

print(f'rows={len(eval_df)}')
print(eval_df.head(3))


In [ ]:
def slugify_model(model_id: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', model_id)


def human_bytes(num: int) -> str:
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    n = float(num)
    for u in units:
        if n < 1024 or u == units[-1]:
            return f'{n:.2f}{u}'
        n /= 1024
    return f'{num}B'


def disk_snapshot(cache_dir: Path, label: str) -> None:
    total, used, free = shutil.disk_usage('/')
    print(f'[{label}] disk_used={human_bytes(used)} free={human_bytes(free)} total={human_bytes(total)}')
    if cache_dir.exists():
        size = sum(p.stat().st_size for p in cache_dir.rglob('*') if p.is_file())
        print(f'[{label}] cache_dir={cache_dir} size={human_bytes(size)}')
    else:
        print(f'[{label}] cache_dir={cache_dir} (not found)')


def get_context_limit(model, tokenizer):
    limit = getattr(model.config, 'max_position_embeddings', None)
    if isinstance(limit, int) and limit > 0:
        return limit
    tok_limit = getattr(tokenizer, 'model_max_length', None)
    if isinstance(tok_limit, int) and 0 < tok_limit < 1000000:
        return tok_limit
    return None


def resolve_batch_size(model_id: str) -> int:
    if 'gemma' in model_id.lower():
        return int(BATCH_SIZE_GEMMA)
    return int(BATCH_SIZE_QWEN)


def build_prompt(source_text: str) -> str:
    return PROMPT_TEMPLATE.format(source_text=source_text)


def clean_hypothesis(text: str) -> str:
    out = (text or '').strip()
    # remove common echoed prefixes
    for prefix in ('<KO>', '<ko>', '번역:', 'Translation:'):
        if out.startswith(prefix):
            out = out[len(prefix):].strip()
    return out


def _load_with_unsloth(model_id: str):
    # Import here so notebook can still open/run partial cells even if unsloth is missing.
    from unsloth import FastLanguageModel, FastVisionModel

    errors = []
    for backend, model_class in (("language", FastLanguageModel), ("vision", FastVisionModel)):
        try:
            unsloth_dtype = torch.bfloat16 if (FORCE_BF16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else None
            model, tokenizer = model_class.from_pretrained(
                model_name=model_id,
                max_seq_length=int(UNSLOTH_MAX_SEQ_LENGTH),
                dtype=unsloth_dtype,
                load_in_4bit=False,
            )
            return model, tokenizer, backend, unsloth_dtype
        except Exception as e:
            errors.append(f"{backend}: {type(e).__name__}: {e}")

    joined = "\n".join(errors)
    raise RuntimeError(f"Unsloth load failed for {model_id}\n{joined}")


def load_model_and_tokenizer(model_id: str, cache_dir: Path):
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['TRANSFORMERS_CACHE'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

    model = tokenizer = None

    is_gemma = 'gemma' in model_id.lower()
    use_unsloth_this_model = USE_UNSLOTH and not (is_gemma and FORCE_TRANSFORMERS_FOR_GEMMA)

    if use_unsloth_this_model:
        if not HAS_UNSLOTH:
            raise RuntimeError('USE_UNSLOTH=True but unsloth is not available. Install unsloth first.')

        unsloth_dtype = None
        loaded = _load_with_unsloth(model_id)
        if isinstance(loaded, tuple) and len(loaded) == 4:
            model, tokenizer, backend, unsloth_dtype = loaded
        else:
            model, tokenizer, backend = loaded
        print(f'  loader=unsloth/{backend} dtype={unsloth_dtype}')

        if backend == 'language' and USE_UNSLOTH_FAST_INFERENCE:
            try:
                from unsloth import FastLanguageModel
                FastLanguageModel.for_inference(model)
                print('  unsloth_for_inference=True')
            except Exception as e:
                print(f'  [WARN] unsloth for_inference failed: {type(e).__name__}: {e}')
    else:
        if is_gemma and FORCE_TRANSFORMERS_FOR_GEMMA:
            print('  loader=transformers (forced for gemma to avoid partial-weight load)')

        model_cfg = AutoConfig.from_pretrained(
            model_id,
            trust_remote_code=True,
            cache_dir=str(cache_dir),
        )
        archs = [a.lower() for a in (getattr(model_cfg, 'architectures', None) or [])]
        use_image_text_cls = HAS_AUTO_MODEL_FOR_IMAGE_TEXT_TO_TEXT and any('conditionalgeneration' in a for a in archs)
        model_cls = AutoModelForImageTextToText if use_image_text_cls else AutoModelForCausalLM
        print(f'  model_class={model_cls.__name__}')

        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            cache_dir=str(cache_dir),
        )

        use_cuda = torch.cuda.is_available()
        if use_cuda:
            if FORCE_BF16:
                if torch.cuda.is_bf16_supported():
                    dtype = torch.bfloat16
                else:
                    print('  [WARN] FORCE_BF16=True but GPU bf16 unsupported; fallback=float16')
                    dtype = torch.float16
            else:
                dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        else:
            dtype = torch.float32

        load_kwargs = dict(
            trust_remote_code=True,
            cache_dir=str(cache_dir),
            torch_dtype=dtype,
            device_map='auto' if use_cuda else None,
            low_cpu_mem_usage=True,
        )

        if use_cuda and ENABLE_FLASH_ATTN_FOR_TRANSFORMERS:
            load_kwargs['attn_implementation'] = 'flash_attention_2'
            print('  attn_implementation=flash_attention_2 (try)')
        elif use_cuda:
            load_kwargs['attn_implementation'] = 'sdpa'
            print('  attn_implementation=sdpa')

        try:
            model = model_cls.from_pretrained(model_id, **load_kwargs)
        except Exception as e:
            if load_kwargs.get('attn_implementation') == 'flash_attention_2':
                print(f'  [WARN] flash_attention_2 failed: {type(e).__name__}: {e}')
                load_kwargs['attn_implementation'] = 'sdpa' if use_cuda else None
                if load_kwargs.get('attn_implementation') is None:
                    load_kwargs.pop('attn_implementation', None)
                print(f"  attn_implementation=fallback_{load_kwargs.get('attn_implementation', 'default')}")
                model = model_cls.from_pretrained(model_id, **load_kwargs)
            else:
                raise

        if use_cuda and ENABLE_XFORMERS_FOR_TRANSFORMERS and HAS_XFORMERS:
            hook = getattr(model, 'enable_xformers_memory_efficient_attention', None)
            if callable(hook):
                try:
                    hook()
                    print('  xformers_memory_efficient_attention=True')
                except Exception as e:
                    print(f'  [WARN] xformers enable failed: {type(e).__name__}: {e}')
            else:
                print('  xformers hook not supported by this model class')

        if not use_cuda:
            model = model.to('cpu')
        print('  loader=transformers')

    # Unsloth may return a Processor-like wrapper; use base tokenizer for encode/decode.
    base_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)
    if base_tokenizer.pad_token_id is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token
    base_tokenizer.padding_side = 'left'

    if hasattr(model, 'config'):
        model.config.use_cache = True
    if hasattr(model, 'generation_config'):
        model.generation_config.use_cache = True

    model.eval()
    device = next(model.parameters()).device
    return model, base_tokenizer, device


def _build_gen_kwargs(tokenizer, effective_max_new: int) -> dict:
    kwargs = {
        'max_new_tokens': int(effective_max_new),
        'do_sample': bool(DO_SAMPLE),
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'use_cache': True,
        'num_beams': 1,
    }
    if DO_SAMPLE:
        kwargs['temperature'] = float(TEMPERATURE)
    return kwargs


def _generate_single(model, tokenizer, device, item_id: str, source_text: str, context_limit: int | None) -> dict:
    prompt = build_prompt(source_text)
    try:
        encoded = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        effective_max_new = MAX_NEW_TOKENS
        if context_limit is not None:
            prompt_len = int(attention_mask.sum().item())
            allowed = context_limit - prompt_len
            if allowed <= 0:
                raise RuntimeError(f'input exceeds context limit: input={prompt_len} limit={context_limit}')
            effective_max_new = min(MAX_NEW_TOKENS, allowed)

        with torch.inference_mode():
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                **_build_gen_kwargs(tokenizer, effective_max_new),
            )

        prompt_len = int(attention_mask.sum().item())
        gen_ids = out[0][prompt_len:]
        decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
        hyp = clean_hypothesis(decoded)
        return {'item_id': item_id, 'source_text': source_text, 'hypothesis': hyp, 'error': ''}
    except Exception as e:
        return {'item_id': item_id, 'source_text': source_text, 'hypothesis': '', 'error': f'{type(e).__name__}: {e}'}


def generate_for_rows(model, tokenizer, device, rows_df: pd.DataFrame, model_id: str) -> list[dict]:
    results = []
    context_limit = get_context_limit(model, tokenizer)
    batch_size = max(1, int(resolve_batch_size(model_id)))

    work_df = rows_df.reset_index(drop=True).copy()
    work_df['_orig_pos'] = work_df.index.astype(int)
    if SORT_BY_INPUT_LENGTH:
        work_df['_sort_len'] = work_df['source_text'].astype(str).str.len()
        work_df = work_df.sort_values(['_sort_len', '_orig_pos'], kind='stable').reset_index(drop=True)

    total = len(work_df)
    print(f'  infer_batch_size={batch_size} sort_by_length={SORT_BY_INPUT_LENGTH}')

    for start in range(0, total, batch_size):
        chunk = work_df.iloc[start:start + batch_size]
        orig_positions = chunk['_orig_pos'].astype(int).tolist()
        item_ids = chunk['item_id'].astype(str).tolist()
        source_texts = chunk['source_text'].astype(str).tolist()
        prompts = [build_prompt(s) for s in source_texts]

        try:
            encoded = tokenizer(
                prompts,
                return_tensors='pt',
                add_special_tokens=False,
                padding=True,
                truncation=False,
                pad_to_multiple_of=8,
            )
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            effective_max_new = MAX_NEW_TOKENS
            if context_limit is not None:
                prompt_lens = attention_mask.sum(dim=1)
                min_allowed = int((context_limit - prompt_lens).min().item())
                if min_allowed <= 0:
                    raise RuntimeError(
                        f'input exceeds context limit in batch: max_prompt={int(prompt_lens.max().item())} limit={context_limit}'
                    )
                effective_max_new = min(MAX_NEW_TOKENS, min_allowed)

            with torch.inference_mode():
                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    **_build_gen_kwargs(tokenizer, effective_max_new),
                )

            for j in range(len(item_ids)):
                prompt_len = int(attention_mask[j].sum().item())
                gen_ids = out[j][prompt_len:]
                decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
                hyp = clean_hypothesis(decoded)
                results.append({
                    'item_id': item_ids[j],
                    'source_text': source_texts[j],
                    'hypothesis': hyp,
                    'error': '',
                    '_orig_pos': orig_positions[j],
                })

        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and len(chunk) > 1:
                print(f'  [OOM] batch {start}:{start+len(chunk)} -> fallback to single')
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                for pos, item_id, source_text in zip(orig_positions, item_ids, source_texts):
                    row = _generate_single(model, tokenizer, device, item_id, source_text, context_limit)
                    row['_orig_pos'] = pos
                    results.append(row)
            else:
                err = f'{type(e).__name__}: {e}'
                for pos, item_id, source_text in zip(orig_positions, item_ids, source_texts):
                    results.append({
                        'item_id': item_id,
                        'source_text': source_text,
                        'hypothesis': '',
                        'error': err,
                        '_orig_pos': pos,
                    })

        except Exception as e:
            err = f'{type(e).__name__}: {e}'
            for pos, item_id, source_text in zip(orig_positions, item_ids, source_texts):
                results.append({
                    'item_id': item_id,
                    'source_text': source_text,
                    'hypothesis': '',
                    'error': err,
                    '_orig_pos': pos,
                })

        done = min(start + len(chunk), total)
        if done % 20 == 0 or done == total:
            print(f'  progress: {done}/{total}')

    results.sort(key=lambda x: x.get('_orig_pos', 0))
    for row in results:
        row.pop('_orig_pos', None)
    return results


def cleanup_model_cache(cache_dir: Path, model=None, tokenizer=None):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    if cache_dir.exists():
        shutil.rmtree(cache_dir, ignore_errors=True)


In [ ]:
# Optional smoke test: first model, first N rows
if RUN_SMOKE_TEST:
    smoke_model = MODELS[0]
    smoke_slug = slugify_model(smoke_model)
    smoke_cache = HF_CACHE_ROOT / smoke_slug
    smoke_out = OUTPUT_DIR / f'{smoke_slug}__smoke.csv'
    smoke_df = eval_df.head(int(SMOKE_LIMIT)).copy()

    print(f'[SMOKE] model={smoke_model} rows={len(smoke_df)}')
    disk_snapshot(smoke_cache, 'smoke_before')

    model = tokenizer = None
    try:
        model, tokenizer, device = load_model_and_tokenizer(smoke_model, smoke_cache)
        smoke_rows = generate_for_rows(model, tokenizer, device, smoke_df, smoke_model)
        pd.DataFrame(smoke_rows)[['item_id', 'source_text', 'hypothesis', 'error']].to_csv(smoke_out, index=False)
    finally:
        cleanup_model_cache(smoke_cache, model=model, tokenizer=tokenizer)
        disk_snapshot(smoke_cache, 'smoke_after')

    print(f'[SMOKE] saved={smoke_out}')
else:
    print('RUN_SMOKE_TEST=False (skip)')



In [ ]:
# Full sequential run (model-by-model download -> infer -> save -> delete)
all_result_frames = []
expected_rows = len(eval_df)

for model_id in MODELS:
    slug = slugify_model(model_id)
    cache_dir = HF_CACHE_ROOT / slug
    per_model_csv = OUTPUT_DIR / f'{slug}__eval120.csv'

    print('\n' + '=' * 100)
    print(f'[MODEL] {model_id}')
    print('=' * 100)

    # Resume-safe: skip if valid existing output exists
    if per_model_csv.exists():
        existing = pd.read_csv(per_model_csv)
        required_cols = {'item_id', 'source_text', 'hypothesis'}
        if required_cols.issubset(set(existing.columns)) and len(existing) == expected_rows:
            print(f'  skip existing file: {per_model_csv}')
            existing = existing[['item_id', 'source_text', 'hypothesis']].copy()
            existing['model_id'] = model_id
            all_result_frames.append(existing)
            continue
        else:
            print(f'  existing output invalid; regenerating: {per_model_csv}')

    disk_snapshot(cache_dir, 'before_load')

    model = tokenizer = None
    try:
        model, tokenizer, device = load_model_and_tokenizer(model_id, cache_dir)
        rows = generate_for_rows(model, tokenizer, device, eval_df, model_id)
        per_df = pd.DataFrame(rows)
        per_df = per_df[['item_id', 'source_text', 'hypothesis', 'error']]
        per_df.to_csv(per_model_csv, index=False)
        print(f'  saved: {per_model_csv}')

        add_df = per_df[['item_id', 'source_text', 'hypothesis']].copy()
        add_df['model_id'] = model_id
        all_result_frames.append(add_df)
    finally:
        cleanup_model_cache(cache_dir, model=model, tokenizer=tokenizer)
        disk_snapshot(cache_dir, 'after_cleanup')

print('\nAll model runs done.')



In [ ]:
# Save merged output + validation checks
merged_csv = OUTPUT_DIR / 'eval120_all_models.csv'

if not all_result_frames:
    raise RuntimeError('No results collected. Check previous cells for failures.')

merged_df = pd.concat(all_result_frames, axis=0, ignore_index=True)
merged_df = merged_df[['model_id', 'item_id', 'source_text', 'hypothesis']]
merged_df.to_csv(merged_csv, index=False)

print(f'saved merged: {merged_csv}')
print(f'total_rows={len(merged_df)}')

per_model_counts = merged_df.groupby('model_id')['item_id'].count().sort_values(ascending=False)
empty_counts = (merged_df['hypothesis'].fillna('').astype(str).str.strip() == '').groupby(merged_df['model_id']).sum()

print('\nrows per model:')
print(per_model_counts)

print('\nempty hypothesis per model:')
print(empty_counts)

expected_per_model = len(eval_df)
bad_models = per_model_counts[per_model_counts != expected_per_model]
if len(bad_models) > 0:
    print('\n[WARN] some models do not have expected row counts:')
    print(bad_models)
else:
    print('\n[OK] all models have expected row counts.')

if HF_CACHE_ROOT.exists():
    leftovers = [p for p in HF_CACHE_ROOT.iterdir() if p.is_dir()]
    if leftovers:
        print('\n[WARN] leftover cache dirs:')
        for p in leftovers:
            print(' -', p)
    else:
        print('\n[OK] cache root has no leftover model dirs.')
